---
numbering:
  enumerator: "26.%s"
  equation:
    template: "%s"
---
(chap_nlp_llms)=
# Large language models for factor investing

<!--
Version: 2.2.12
Last updated: 2026-07-21
Provenance: fourth chapter of Part V of the 2nd edition, extracted from the
monolithic Chapter_20_Draft19.md (v3.9.7, archived in
`_archive_monolithic_v3.9.7/`).
-->


## Introduction

The pre-trained language models of Chapter 22 close the contextual gap of the embeddings of Chapter 21 but operate on a fixed-size feature extraction paradigm: a BERT-class encoder produces a vector representation that is fed into a downstream classifier or regressor. Large language models break with this paradigm on two fronts. They are vastly larger (billions to trillions of parameters trained on internet-scale corpora), and they are typically *prompted* to produce interpretable structured output directly rather than asked to emit a feature vector. The empirical consequence is that LLMs can perform multi-step reasoning, follow instructions, synthesise across documents, and use tools, opening factor-investing applications that the encoder-only paradigm of Chapter 22 cannot reach: agentic equity research, multi-axis sentiment scoring, automated thematic classification, and retrieval-augmented analysis of long-form financial documents at scale.

## Model families and capabilities

The classical pipelines of Chapter 20 treated text through fixed stages: tokenize, count, weight, classify. The methods worked, but they sacrificed two things that the financial document genre values most. They threw away discourse structure (a 10-K paragraph cannot be reconstructed from its word counts), and they could not generalize across phrasing variants without expensive supervised retraining for each task. Beginning with BERT [@devlin2019bert] and accelerating sharply after GPT-3 [@brown2020language], transformer-based language models inverted both constraints. A single pretrained model now serves as the substrate for sentiment classification, summarization, entity extraction, question answering, code generation, and free-form analyst-style commentary, with task-specific behaviour induced through prompting or lightweight fine-tuning rather than full retraining.

Before turning to the algorithmic mechanics that make these models work (scaling laws, alignment, retrieval, and agents, developed in the sections that follow), it is useful to map the practical landscape, because an applied-finance reader benefits from an operational map of *what* is available and *at what cost* before working through the details of *why*. The five paragraphs below sketch the model families that quantitative practitioners encounter in mid-2026, organized by vendor, followed by a comparison table that summarises the dimensions that matter for an investment-research deployment: open versus closed weights, context window, parameter scale, native multimodality, reasoning specialization, and approximate inference cost. We then close with a short note on finance-specific models, which sit alongside the general-purpose families and form a distinct but smaller cluster.

**OpenAI: GPT-5.6 family.** OpenAI continues to set the API-quality benchmark for finance-style instruction following; the GPT-4 technical report [@achiam2023gpt4] remains the canonical public reference for the architecture of the flagship line, whose more recent successors are no longer documented in comparable detail. The flagship as of mid-2026 is the GPT-5.6 family (released July 2026), a multimodal model with a context window on the order of one million tokens, comfortably large enough to ingest a full 10-K with its exhibits. Reasoning is now exposed as a tunable *effort* level (from low through xhigh) rather than as a separate model, so a single endpoint spans fast single-shot answers and extended chain-of-thought deliberation, with cost and latency scaling in the effort chosen. The models are closed-weight (no on-premise deployment), parameter counts are undisclosed, and pricing is metered per million tokens with separate input and output rates. For financial workflows the high-effort tiers are noticeably stronger on multi-step earnings-call Q&A and on legal-style numerical reasoning over filings, but the cost premium over a low-effort call can be an order of magnitude or more for the same prompt.

**Anthropic: Claude Opus 4.8, Sonnet 5, Haiku 4.5.** Anthropic's Claude family is structured as a tiered ladder: Opus (largest, most capable), Sonnet (balanced), Haiku (fastest, cheapest). The current line spans Claude Opus 4.8, the Claude 5 generation (Sonnet 5 and the lightweight Fable 5), and Claude Haiku 4.5. The Opus tier offers a context window up to one million tokens, which fundamentally changes the design of RAG pipelines for long documents: an entire annual report, with its exhibits and footnotes, fits in a single prompt without retrieval-shard engineering, and an extended-thinking mode adds reasoning depth on demand. Claude is closed-weight and competitively priced at the Haiku tier, where finance applications such as transcript triage and ticker-level news summarization at scale become economically viable. Its training-data and tooling emphasis on long-document handling make it a natural choice for filings-heavy workflows.

**Meta: Llama 4 series.** Meta's Llama family is the dominant open-weights line. The Llama 4 series, built on native sparse-mixture-of-experts routing with context handling extended into the multi-million-token range, is the most widely deployed open model in the finance community as of mid-2026, having succeeded the Llama 3.3 70B model that anchored on-premise deployments through 2025. Because weights are openly released under a permissive licence, Llama models can be deployed on-premise (a requirement at many regulated firms), fine-tuned domain-specifically (FinGPT; [@yang2023fingpt]), and quantized to run on commodity GPUs using techniques such as LoRA [@hu2022lora] and QLoRA [@dettmers2023qlora]. The trade-off is that out-of-the-box quality on the hardest reasoning tasks still trails the closed flagships by a measurable margin, though the gap has narrowed with each model generation.

**DeepSeek: V3.2, V4 series.** DeepSeek (China) reached parity with the closed flagships on several reasoning suites at a fraction of the inference cost, first with its V3 and R1 releases of 2024-2025 and, as of mid-2026, with the stable V3.2 (a 671-billion-parameter mixture-of-experts model with 37 billion active parameters per token and a 128K context, priced aggressively below the closed reasoning models) and the newer, much larger V4 series previewed under a permissive licence. Both are open-weight. For quantitative finance teams the appeal is concrete: a self-hosted DeepSeek instance handles complex earnings-call Q&A and multi-document filings comparisons at roughly an order-of-magnitude lower marginal cost than the closed reasoning models, with the obvious caveat that geopolitical and supply-chain considerations attach to the weights' origin.

**Mistral: Mistral Large 3, Codestral.** Mistral (France) anchors the European open-weights ecosystem. Its flagship, Mistral Large 3, is a large open-weight sparse-mixture-of-experts model (roughly 41 billion active of 675 billion total parameters, released under a permissive licence with a context window in the mid-hundreds of thousands of tokens), occupying the open-deployment niche between Llama and DeepSeek, with strong multilingual performance, including European languages relevant to non-US factor research. Codestral specialises in code generation, which is operationally relevant for the agentic-research workflows discussed in Section 23.11.

**TABLE 23.1: Model landscape for financial NLP applications (as of mid-2026).** Specifications are approximate, change rapidly, and are presented for ordinal comparison rather than benchmarking. The "Strengths for finance" column distils properties documented in vendor technical reports and the practitioner literature.

| Family | Vendor | Flagship (mid-2026) | Weights | Context (tokens) | Approx. params | Reasoning variant | Strengths for finance |
|---|---|---|---|---|---|---|---|
| GPT | OpenAI | GPT-5.6 family | Closed | up to ~1M | Undisclosed | effort levels (low-xhigh) | API quality, broad instruction-following |
| Claude | Anthropic | Opus 4.8, Sonnet 5, Haiku 4.5 | Closed | up to 1M | Undisclosed | Extended thinking | Long-document ingestion, tiered cost ladder |
| Llama | Meta | Llama 4 series | Open | 128K to multi-M | 17B active+ (MoE) | (community fine-tunes) | On-premise deployment, FinGPT base |
| DeepSeek | DeepSeek (CN) | V3.2, V4 series | Open | 128K | 37B active / 671B total (V3.2) | R-series | Cost-efficient reasoning, open MoE |
| Mistral | Mistral (FR) | Mistral Large 3 | Open | up to ~256K | 41B active / 675B total | (Mistral reasoning) | EU residency, multilingual, Codestral |
| Gemini | Google | Gemini 3, 3.5 Pro | Closed | up to 2M | Undisclosed | Deep Think | Multimodality (charts, video), tool use |
| Qwen | Alibaba | Qwen 3.7 Max, Qwen 3.6 | Mixed | up to ~1M | to 400B+ (MoE) | Qwen reasoning | Chinese-language news, A-share coverage |
| Domain | (various) | FinBERT, FinGPT, BloombergGPT | Mixed | 512 to 32K | 110M to 50B | n/a | Pre-tuned to financial lexicon |

TABLE 23.1 highlights several practical patterns. *Open versus closed* is the most consequential axis for institutional deployments: open weights enable on-premise inference (required for sensitive prompts that include unreleased internal research), fine-tuning on proprietary corpora (a private earnings-call dataset, for example), and protection against vendor lock-in; closed weights deliver higher out-of-the-box quality and a managed cost-per-token. *Context window* directly determines whether retrieval-augmented generation [@lewis2020retrieval] is required: with a one-million-token window a Claude Opus 4.8 or Llama 4 prompt can hold an entire 10-K plus the prior year's filing for direct comparison, eliminating the need for retrieval shards; with a 32K window, RAG is mandatory for any document-level reasoning. *Parameter count* is becoming a less useful capability proxy than it was during the dense-model era (2020-2023): mixture-of-experts models such as Mistral Large 3 and DeepSeek V3.2 activate only a small fraction of their parameters per token, so headline parameter counts misrepresent both inference cost and capability. *Reasoning variants* (the GPT-5.6 high-effort tiers, DeepSeek's R-series, Gemini Deep Think, Claude extended thinking) systematically outperform their base counterparts on multi-step financial calculation tasks but at several times the inference cost; they are usually deployed sparingly, on the small subset of prompts where multi-step reasoning is the bottleneck rather than retrieval or summarization.

A separate cluster of *finance-specific* models sits alongside the general-purpose families (FinBERT, BloombergGPT, FinGPT, PIXIU, InvestLM, and others); we defer the deep treatment to Section 23.5 and note here only that these domain models are most valuable as components of larger pipelines (a FinBERT sentiment head feeding a GPT-5.6 or Claude Opus 4.8 reasoning step, for example) rather than as drop-in replacements for the flagship general models.

Throughout the sections that follow, we separate *what the models can do* (their capabilities at the time of writing) from *what is durable* (the algorithmic and economic forces that drive the field), since the specific entries in TABLE 23.1 will be out of date within months while the structural lessons should age more gracefully.

## The inference ecosystem: serving, batching, and cost

The model landscape of Section 23.2 tells us *what* is available; the inference ecosystem tells us *how* and *at what cost*. For a quantitative-research workflow, the difference between two serving choices can be a factor of ten in dollars per million tokens and a factor of one hundred in latency, so the choice deserves the same systematic treatment we apply to model selection. We organize the ecosystem along four layers: specialized inference hardware, cloud inference providers, hyperscaler-integrated APIs, and on-premise self-hosting; we then close with two cross-cutting cost levers (batch APIs and prompt caching) that are decisive for large-corpus financial NLP pipelines.

![FIGURE 23.1: The LLM inference ecosystem for financial NLP](images/figure_23_1_llm_ecosystem.png)

FIGURE 23.1 organizes the discussion that follows: read top-to-bottom, an application calls an API endpoint, which is served by a runtime, which executes on silicon; read bottom-to-top, capability flows up from the chip to the analyst. The two horizontal boxes inside Layer 1 (batch API, prompt caching) cut across every provider and every runtime and are the dominant cost levers in deployed financial NLP pipelines.

**Specialized inference accelerators.** A new generation of hardware vendors built chips specifically for transformer inference rather than training. *Groq* (founded 2016) ships Language Processing Units (LPUs) that achieve 500 to 800 output tokens per second on Llama 70B with a time-to-first-token under ten milliseconds, an order of magnitude faster than commodity GPU serving. *Cerebras* offers wafer-scale-engine (WSE-3) inference at speeds reported above 2,000 tokens per second on the same model family, available through the Cerebras Cloud and as an on-premise appliance; Meta partnered with Cerebras to serve Llama at headline-grabbing latencies. *SambaNova* (SN40L) targets the very largest open models (Llama 405B and beyond) at competitive cost-per-token. For financial use cases the latency premium of these accelerators matters most in interactive analyst workflows (real-time earnings-call Q&A, intra-meeting summarization) and in streaming-news-classification pipelines where end-to-end latency is the binding constraint; for batch document processing the throughput-per-dollar advantage of conventional GPU clusters is typically larger.

**Cloud inference providers.** A second tier of vendors specializes in serving open-weight models on managed GPU clusters: *Together AI*, *Fireworks AI*, *Anyscale*, *Replicate*, *Lepton AI*, *Modal*, and *Hugging Face Inference Endpoints* are the most prominent in 2026. They host Llama, Mistral, DeepSeek, and Qwen weights on H100 and H200 hardware, expose OpenAI-compatible REST APIs, and price by tokens consumed; published rates are typically 30 to 60% below the closed-flagship equivalents at comparable benchmark quality. Operationally, these providers are the default choice when a team has settled on an open-weights model but lacks the engineering bandwidth to run a self-hosted serving stack.

**Hyperscaler-integrated APIs.** Regulated institutions usually consume models through the cloud they already use: *AWS Bedrock* aggregates Claude, Llama, Mistral, and Titan behind a single IAM-protected endpoint; *Azure OpenAI Service* mirrors the OpenAI model catalogue with Microsoft's compliance wrapper; *Google Vertex AI* serves Gemini natively and brokers Anthropic and Llama models. These integrations add a modest per-token premium (typically 5 to 15%) but deliver enterprise features (private VPC routing, audit logs, regional residency such as eu-central-1 or asia-northeast-1, key-management integration) that are mandatory under most bank IT-security frameworks. For European or Asian factor-research teams the regional-residency dimension is often a deciding factor on its own.

**On-premise self-hosting.** The fourth layer is the team's own GPU fleet. *vLLM* [@kwon2023pagedattention] is the dominant open-source serving runtime, with continuous batching, paged KV-cache memory management, and OpenAI-compatible endpoints; it is the de facto choice for production-scale internal deployment of Llama, Mistral, and DeepSeek models on A100 or H100 fleets. *Hugging Face Text Generation Inference (TGI)* is an alternative with similar scope. *Ollama* and *LM Studio* target developer laptops and prototyping. *llama.cpp* enables 70-billion-parameter inference on commodity hardware (a Mac Studio with sufficient unified memory, or two A100s with INT4 quantization), which is useful for confidential pilots that cannot leave a controlled environment. Self-hosting eliminates per-token vendor charges but introduces capital expenditure on GPUs and ongoing engineering cost; the break-even against cloud inference depends on sustained utilization and typically arrives around 10 to 30 billion tokens served per month for a 70B-class model.

**Batch APIs.** For NLP pipelines that ingest large historical corpora (5 to 50 million news articles, every 10-K filing in EDGAR since 1994, the full Russell 3000 earnings-call history), synchronous APIs are wasteful. The major providers therefore offer *batch* endpoints with explicit asynchronous semantics and a discounted price. *OpenAI Batch API* accepts JSONL submissions, returns results within 24 hours, and charges 50% of the standard rate. *Anthropic Message Batches* offers the same 50% discount. *Google Vertex AI Batch Prediction* serves the same role for Gemini. The economic implication is direct: a one-million-document sentiment-scoring run that costs USD 20,000 at synchronous rates becomes USD 10,000 in batch, which is often the difference between an exploratory factor and a production one. Batch endpoints are the right default for backtest construction, periodic factor recomputation, and any workload where 24-hour latency is acceptable.

**Prompt caching.** A second cross-cutting cost lever exploits the structure of long-context financial workloads. When a pipeline repeatedly sends prompts that share a long static prefix (a 10-K document plus a varying question template, for example), most providers will cache the encoded prefix and re-bill only the variable suffix. *Anthropic prompt caching* discounts cached input tokens by roughly 90%; *OpenAI* offers an equivalent feature on its long-context models; *Google Vertex* exposes context caching as a first-class object with a user-defined retention period. For RAG-over-corpus pipelines, prompt caching can collapse the unit economics of long-context inference by an order of magnitude and is the single most underused cost lever in deployed financial NLP systems.

**Practical synthesis.** A mature factor-research deployment in 2026 rarely commits to a single provider. A typical stack we have seen in practice combines: (i) a quantized FinBERT or Llama-7B-derived classifier on a self-hosted vLLM cluster for cheap high-volume sentiment scoring of news flow; (ii) Claude Sonnet 5 on AWS Bedrock with prompt caching enabled for routine analyst-style summarization over filings and transcripts; (iii) the OpenAI Batch API or Anthropic Message Batches for one-shot historical-corpus reprocessing; (iv) GPT-5.6 or Claude Opus 4.8 reasoning variants used sparingly and via the synchronous API, restricted to the hardest 1 to 5% of queries where multi-step reasoning is the bottleneck; and (v) Groq or Cerebras for the rare interactive workflow that requires sub-100-millisecond response times. The optimization is across providers and modes rather than within any one, and the resulting cost reductions over a naive "one-flagship-for-everything" baseline are routinely 80 to 95% with no measurable degradation in research output.

## What makes an LLM "large"

The models discussed so far (BERT, FinBERT) have between 110 million and 340 million parameters. *Large language models* (LLMs) scale this by orders of magnitude: GPT-3 [@brown2020language] has 175 billion parameters, and more recent models are larger still. This quantitative increase leads to qualitative changes in capability, a phenomenon known as *emergence*.

@kaplan2020scaling empirically established *scaling laws* that relate model performance to three factors: the number of parameters $N$, the size of the training dataset $D$, and the compute budget $C$. The key finding is that the test loss decays predictably as a power law in each factor when the other two are not the bottleneck,

$$
L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N} + L_\infty, \qquad L(D) \approx \left(\frac{D_c}{D}\right)^{\alpha_D} + L_\infty
\tag{23.1}
$$

with empirically estimated exponents $\alpha_N \approx 0.076$ and $\alpha_D \approx 0.095$ for autoregressive transformers on natural-language data, and irreducible loss $L_\infty$ bounding the achievable test error. Equation (23.1) is the algebraic statement behind the scaling race that produced GPT-3, PaLM, LLaMA, and their successors, and it explains why doubling the parameter count of a frontier model is expected to deliver a sub-linear but predictable improvement rather than a saturation. Subsequent work refined the exponents and introduced *compute-optimal* scaling, in which $N$ and $D$ should grow at comparable rates rather than scaling $N$ in isolation.

At sufficient scale, LLMs acquire capabilities that are absent or unreliable in smaller models:

- **Few-shot learning.** The model can perform a new task given only a handful of examples in the prompt, without any parameter updates.
- **Chain-of-thought reasoning.** When prompted to "think step by step," large models can decompose complex problems into intermediate steps.
- **Instruction following.** After alignment training (see Section 23.5), LLMs can follow natural language instructions to perform tasks they were not explicitly trained on.

## Key techniques

Several techniques underpin the capabilities of modern LLMs.

**Tokenization.** LLMs do not operate on words directly. Instead, they use sub-word tokenization schemes such as Byte-Pair Encoding (BPE), introduced by @sennrich2016neural, or SentencePiece [@kudo2018sentencepiece]. These algorithms split rare words into frequent sub-word units (e.g., "restructuring" might become "re" + "structur" + "ing") while keeping common words intact. The vocabulary size is typically 30,000 to 100,000 tokens.

**Decoding strategies.** When generating text, the model produces a probability distribution over the vocabulary at each position. *Greedy decoding* selects the highest-probability token at each step. *Beam search* maintains multiple hypotheses. *Nucleus sampling* [@holtzman2020curious] samples from the smallest set of tokens whose cumulative probability exceeds a threshold $p$ (typically 0.9 or 0.95), balancing diversity and coherence.

**Prompting.** The most distinctive feature of LLMs is their ability to perform tasks through *prompting*, where the task specification is provided as natural language input:

- *Zero-shot:* "Classify the following headline as positive, negative, or neutral: [headline]"
- *Few-shot:* Provide several labeled examples before the target, so the model can infer the pattern.
- *Chain-of-thought:* "Classify the following headline. First, identify the key financial terms. Then, assess whether the overall tone is positive or negative. Finally, provide your classification."

**Reasoning at inference time.** A distinct line of recent research treats the reasoning trajectory itself as something to be elicited and shaped. @wei2022cot showed that simply asking a sufficiently large model to "think step by step" before answering, a technique now known as chain-of-thought (CoT) prompting, materially improves accuracy on arithmetic, commonsense, and symbolic-reasoning benchmarks. @wang2023selfconsistency extended this to *self-consistency*: sampling multiple CoT trajectories and majority-voting the final answers raises accuracy by another margin at a multiple of the inference cost. @yao2023treeofthoughts further generalised the idea to *tree of thoughts*, where the model explores a branching deliberation tree rather than a single linear chain. Most recently, a family of *reasoning-tuned models* (OpenAI o1 and o3; the DeepSeek-R1 release of @guo2025deepseekr1, which trains a reasoning core via reinforcement learning on verifiable rewards) internalises the CoT pattern during training, producing models that spend computation generating private chain-of-thought traces before emitting an answer. For financial workloads these techniques matter because multi-step reasoning (combining a sentiment score with an earnings surprise and a sector-rotation prior, for example) is exactly the regime in which CoT and reasoning-tuned models open the largest gap over flat single-shot models.

**Parameter-efficient fine-tuning.** Full fine-tuning of a 70-billion-parameter model is impractical for most teams. LoRA [@hu2022lora] replaces the gradient update of a weight matrix $\mathbf{W}_0 \in \mathbb{R}^{d \times k}$ with a low-rank residual,

$$
\mathbf{W} = \mathbf{W}_0 + \mathbf{B}\mathbf{A}, \quad \mathbf{A} \in \mathbb{R}^{r \times k}, \quad \mathbf{B} \in \mathbb{R}^{d \times r}, \quad r \ll \min(d, k)
\tag{23.2}
$$

where only $\mathbf{A}$ and $\mathbf{B}$ are trained while $\mathbf{W}_0$ is frozen. The number of trainable parameters drops from $d \cdot k$ to $r(d + k)$, which is typically a factor of 100 to 1{,}000 smaller. QLoRA [@dettmers2023qlora] extends the trick to 4-bit-quantized base weights, bringing the fine-tuning of a 70B-parameter model into the memory budget of a single consumer GPU.

**Alignment.** Raw pre-trained LLMs optimize next-token prediction, which does not guarantee helpful, truthful, or safe outputs. *Reinforcement learning from human feedback* (RLHF), introduced by @ouyang2022training, trains a reward model $r_\phi(x, y)$ from human pairwise preferences over completions and then fine-tunes the policy $\pi_\theta$ by maximizing the expected reward minus a Kullback-Leibler penalty that keeps the policy close to a frozen reference model $\pi_{\text{ref}}$,

$$
\max_{\pi_\theta}\;\mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi_\theta(\cdot \mid x)}\!\left[r_\phi(x, y)\right] \;-\; \beta\, \mathbb{E}_{x \sim \mathcal{D}}\!\left[\text{KL}(\pi_\theta(\cdot \mid x)\,\|\, \pi_{\text{ref}}(\cdot \mid x))\right].
\tag{23.3}
$$

Direct preference optimization, introduced by @rafailov2023direct, reparameterizes Equation (23.3) so that the reward model and the policy collapse into a single closed-form supervised objective on the preference data, eliminating the unstable reinforcement-learning loop. Alignment is critical for financial applications where factual accuracy and consistency are paramount.

## Finance-specific LLMs

**BloombergGPT** [@wu2023bloomberggpt] is a 50-billion parameter model trained on a mix of general text (345 billion tokens) and Bloomberg's proprietary financial data (363 billion tokens). It outperforms general-purpose models of similar size on financial NLP benchmarks (sentiment analysis, named entity recognition, financial question answering) while maintaining competitive performance on general tasks. The model is not publicly available, limiting its use outside Bloomberg.

**Open-source alternatives** have narrowed the gap. Models such as LLaMA [@touvron2023llama] and Mistral [@jiang2023mistral] can be fine-tuned on financial data to create domain-specific models at a fraction of the cost of training from scratch. FinGPT [@yang2023fingpt] provides an open-source framework for financial LLM development, including data pipelines, fine-tuning scripts, and evaluation benchmarks.

As an early and self-aware illustration of what general-purpose LLMs can add to the discussion, @zaremba2023chatgpt wrote a short communication for *Modern Finance* almost entirely by prompting ChatGPT, asking it to enumerate financial NLP use cases (sentiment scoring, text classification, summarisation, fraud detection, regulatory compliance) and to reflect on its own limitations. The paper is interesting more for what it is (a published article drafted by the model it surveys) than for the novelty of its conclusions, which echo the standard warnings about hallucination, training-data bias, and exposure to GDPR and MiFID II when feeding client-specific text into third-party APIs. As a bridge into the more substantive comparisons that follow, it is useful mainly to remind ourselves that "LLMs can write about themselves" is not the same as "LLMs generate alpha," and that the burden of proof for the latter sits with empirical evaluations of the kind we review next.

A direct challenge to the "bigger is better" reading of the scaling literature is provided by @rahimikia2024revisiting, who pit FinText, a 124.65-million-parameter RoBERTa pre-trained year by year on financial text, against general-purpose 7-8 billion-parameter models (Llama 3, Mistral) on US equity return prediction over 2017-2023. On an equal-weighted long-short portfolio sorted on the model-derived sentiment score, FinText delivers an annualised Sharpe ratio of 3.76 against 3.20 for Llama 3, with roughly 64 times fewer parameters. A second, methodological contribution of the paper is worth flagging: by projecting the model embeddings into two dimensions via PCA, the authors show that the general-purpose LLMs visibly cluster around known events (COVID in 2020, the launch of GPT-3.5 in late 2022, the Trump-related news of 2019), suggesting some degree of temporal contamination in the *representation*. This contamination does not translate into improved classification or portfolio performance, which is reassuring, but it is a warning for practitioners who use off-the-shelf LLM embeddings as features in backtests that span training-window boundaries.

We do not have FinText to hand, but Chapter 21's evaluation on the book's NLP US universe already tests the more basic version of the claim: does a compact finance-specific encoder produce a genuinely tradeable signal? TABLE 21.1 shows that it does at short horizons. Scored on the analyst-Q&A segment at the five-day horizon, FinBERT (a roughly 110-million-parameter finance-specific encoder) posts an information coefficient of 0.074, with an information ratio of 1.21 and a Newey-West t-statistic of 10.7, the strongest contextual-encoder entry in that table. TABLE 21.2 adds the complementary point that this signal is close to orthogonal to the Fama-French and Carhart six-factor stack (R² of 0.03), so it is not a re-skinning of momentum or quality; like almost every sentiment method, however, its monthly long-short alpha is not statistically distinguishable from zero once the factor stack absorbs the systematic component, a reminder that the FinBERT edge is a short-horizon one. This is the concrete counterpart to the @rahimikia2024revisiting result: genuine signal from a model three orders of magnitude smaller than a frontier LLM.

On the same data the large four-axis LLM composite of Section 23.8 posts both a higher raw information coefficient than FinBERT (0.188 in TABLE 21.1) and the only right-signed, marginally significant monthly alpha in TABLE 21.2 (about 6% per year, Newey-West t of 1.70). That raw comparison is exactly the naive, in-sample-favouring one that Rahimikia and Drinkall's year-by-year design is built to correct, and the look-ahead caveats discussed later in this chapter apply to the composite in full, since Llama-3.1-8B's training corpus overlaps the evaluation window. The durable lesson is not that the small model always wins, but that a compact specialist clears the bar for a deployable short-horizon signal at a tiny fraction of the cost.

The choice between a general-purpose LLM, a finance-specific LLM, and a fine-tuned open-source model therefore depends on the application. For sentiment classification on standard financial text, fine-tuned FinBERT (Chapter 22) or a compact domain-specialised RoBERTa in the spirit of @rahimikia2024revisiting is often sufficient and far more cost-effective. For more complex tasks requiring reasoning, summarization, or handling novel language, a larger LLM is needed.

## Retrieval-augmented generation

A limitation of LLMs is that their knowledge is frozen at the time of training. A model trained before 2024 knows nothing about events in 2025. *Retrieval-augmented generation* [@lewis2020retrieval] addresses this by grounding the model's output in external documents retrieved at query time. Formally, RAG decomposes the predictive distribution into a retriever and a generator,

$$
p(y \mid x) = \sum_{z \in \mathcal{Z}} p_\eta(z \mid x)\, p_\theta(y \mid x, z),
\tag{23.4}
$$

where $x$ is the user query, $z$ ranges over candidate documents (typically the top-$k$ chunks returned by a vector search over an indexed corpus), $p_\eta(z \mid x)$ is the retriever's score, and $p_\theta(y \mid x, z)$ is the LLM-generator's likelihood conditioned on the retrieved passage. In practice the sum is truncated to the top-$k$ retrieved documents and either marginalized at the token level (RAG-Sequence) or used as additional context to a single generator pass.

The RAG pipeline works as follows:

1. The user provides a query (e.g., "What are the key risks mentioned in Apple's 2024 10-K filing?").
2. A retrieval system searches a document index and returns the most relevant passages.
3. The retrieved passages are prepended to the query as context for the LLM.
4. The LLM generates a response grounded in the retrieved text.

For financial applications, RAG is particularly valuable because it allows the model to answer questions about specific documents (a particular 10-K filing, a recent earnings call transcript) without requiring fine-tuning on those documents. It also reduces *hallucination* (the generation of plausible but factually incorrect text), because the model's answer is anchored in actual source text.

The choice of retrieval strategy matters considerably in finance, where documents are long, cross-referenced, and often rich in structured relationships (entities, subsidiaries, product lines). @sarmah2024hybridrag introduce HybridRAG, which combines the two dominant retrieval paradigms: vector-based RAG (dense similarity search over embedded text chunks) and knowledge-graph-based RAG (graph traversal over entity relationships extracted from the corpus). Evaluated on a corpus of earnings-call transcripts using finance Q&A benchmarks, HybridRAG improves both answer faithfulness and retrieval precision relative to either pure-vector or pure-graph retrieval alone. The intuition is that vector search excels at finding passages with similar phrasing but may miss relevant cross-entity links, while knowledge-graph traversal captures structured relationships that are underrepresented in dense embeddings; combining the two retrieval signals is therefore complementary rather than redundant. For practitioners building RAG pipelines over large corpora of 10-K filings or transcripts, HybridRAG represents the current state of the art and a useful engineering blueprint. A complementary and highly practical result concerns the retrieval step's most mundane parameter, the chunk. @jimenoyepes2024chunking show that chunking financial filings by *structural element* (item, table, section) rather than by fixed token windows yields near-optimal chunk sizes with no tuning and materially improves retrieval quality on financial documents, a reminder that in production RAG pipelines the document-preprocessing choices often matter as much as the choice of retriever or generator.

## Sentiment and tone extraction via prompting

One of the most impactful applications of LLMs in finance is using them as sophisticated sentiment extractors that go far beyond dictionary methods. Rather than counting words, an LLM can be prompted to assess the overall tone of a text, taking into account context, negation, conditionals, and nuance.

@kim2024transcripts prompt GPT to read earnings-call transcripts and generate risk summaries and assessments, producing firm-level risk measures that dictionary methods cannot recover. Their central finding is that these GPT-based measures carry significant incremental information content: they outperform established dictionary-based risk measures in predicting firm-level (abnormal) return volatility and in explaining corporate decisions such as investment and innovation. The model captures nuances such as hedging language, conditional statements, and the framing of forward-looking risks that word-counting methods miss.

@lopezlira2023chatgpt applied ChatGPT to classify financial news headlines as positive or negative and found that the resulting sentiment scores predicted next-day stock returns with a Sharpe ratio exceeding 5 in certain configurations. While this result is likely inflated by the small sample and idealized assumptions (zero transaction costs, instantaneous execution), it demonstrates the potential of LLM-based sentiment for return prediction.

A larger and peer-reviewed test of the same idea is provided by @kirtac2024sentiment, who score 965,375 US news articles from 2010 to 2023 with a GPT-class model and benchmark it against BERT, FinBERT, and the Loughran-McDonald dictionary. The LLM sentiment scores predict next-day returns with a statistically and economically significant edge, and a long-short portfolio sorted on them outperforms the dictionary-based equivalent. The value of the study for our purposes is precisely its scale: it shows that the LLM's advantage over lexicon methods survives on nearly a million articles rather than being an artefact of the small, idealized samples that make results like Lopez-Lira and Tang's difficult to bank on.

**A worked example of cross-sectionally calibrated prompting.** Because LLM-derived scores in finance are typically used in *cross-sectional* sorts, where the absolute level of a score matters only relative to other firms in the same period, the single most important design choice in the prompt is the *anchor*: the explicit statement of what a score of zero represents. Without an anchor, off-the-shelf LLMs default to a "things are pretty good" calibration that produces a compressed positive distribution with little cross-sectional dispersion. To make this concrete, the case study of Chapter 24 defines the following system prompt, held in the variable `SYSTEM_PROMPT` that the scoring call further below reuses, to score earnings-call excerpts on four dimensions through the Groq Llama-3.3-70B API:

In [ ]:
SYSTEM_PROMPT = """You are a senior sell-side equity analyst with 15 years of experience scoring
the tone of public-company earnings call transcripts. You score excerpts on
four dimensions, each on a continuous scale from -1.0 to +1.0, calibrated so
that average S&P 500 prepared remarks score 0.0 on every dimension.

DIMENSIONS
1. managerial_confidence: how confident management sounds about the FORWARD outlook.
   +1.0 = raising guidance with explicit conviction, firm numerical targets.
    0.0 = reaffirming guidance with normal hedging.
   -1.0 = visibly cautious, withdrawing guidance, persistent hedging on outlook.
2. specificity: numerical, named-product, segment-level granularity vs vague framing.
3. surprise_negative: tone shift suggesting bad news the market may not yet have priced.
4. macro_vs_firm_risk_framing: when risk is discussed, where management points the blame
   (firm-specific = +1, macro/external = -1).

CALIBRATION RULES
- Anchor on the population, not the excerpt. Average S&P 500 prepared remarks
  should be near 0 on every dimension. A score of 0 means "indistinguishable
  from a typical large-cap quarterly call."
- Reserve |score| > 0.5 for clearly above-average signal.
- Do not penalize standard safe-harbor language as hedging; it is universal.

OUTPUT: strict JSON with the four scores plus a one-sentence rationale and a
regime_tag from {guidance_raise, guidance_cut, in_line, transition,
crisis_response}."""

Two design choices in this prompt are worth highlighting. First, the *calibration anchor* ("average S&P 500 prepared remarks should be near 0") is the single most consequential sentence: dropping it changes the cross-sectional standard deviation of the LLM scores by roughly a factor of three on the validation sample, while changing the mean of the distribution by relatively little. Second, the *regime tag* is a low-cost side-output that turns each LLM call into a categorical conditioning variable: it allows the chapter's case study (Chapter 24) to report that managerial confidence predicts forward returns conditional on `in_line` or `transition` regimes but not in `crisis_response` quarters, exactly the regime-dependent pattern that @rutanen2020factor document at the factor-momentum level.

The same tools extend naturally to social media text. @albladi2025twitter provide a systematic review of Twitter sentiment analysis across the NLP literature and report that transformer-based classifiers (BERT, RoBERTa, GPT) now routinely exceed 90% accuracy on standard Twitter benchmarks, a clear step change from the dictionary and classical-ML baselines that dominated before 2018. For financial applications, however, several caveats carry over: the 280-character format truncates context that LLMs rely on for disambiguation; slang, hashtags, and emojis are underrepresented in financial pre-training corpora; sarcasm and code-switching remain open problems; and the class distribution is typically imbalanced, with a long tail of neutral tweets. These are not minor annoyances, they directly interact with the coverage and selection biases that we return to in Section 23.13, and they explain why social-media-based factors tend to have the highest headline Sharpe ratios *and* the widest range of replicated performance.

The practical advantage of LLM-based sentiment over dictionary methods is the ability to handle *implicit* sentiment. A statement like "We expect revenue growth to moderate as we invest heavily in long-term R&D initiatives" is neither positive nor negative by dictionary measures, but an LLM can recognize the forward-looking investment narrative and classify the tone as cautiously optimistic.

To make this concrete, we score a single earnings-call excerpt with the four-dimension prompt above by calling Llama-3.3-70B through the Groq API. The request is deliberately ordinary: a *system* message carrying the calibrated analyst instructions, a *user* message carrying the transcript text, and a few decoding parameters.

In [ ]:
from groq import Groq                          # pip install groq; GROQ_API_KEY in the environment
client = Groq()                                 # Groq exposes an OpenAI-compatible endpoint

# Two messages: a fixed SYSTEM role (the four-dimension prompt shown above, identical for every
# call) and a per-call USER role carrying light metadata plus the transcript excerpt to be scored.
excerpt = load_bucket_text('KO', '2006-Q3', 'prep_mgmt')     # ~3,000 words of prepared remarks
user_msg = (f"Ticker: KO   Quarter: 2006-Q3   Section: prepared remarks (management)\n\n"
            f"Excerpt:\n\n{excerpt}")

We send the pair to the model, holding the decoding parameters fixed so the scores are reproducible across runs.

In [ ]:
resp = client.chat.completions.create(
    model='llama-3.3-70b-versatile',            # ### other models: see the swaps below
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},   # role instructions, held fixed across calls
        {'role': 'user',   'content': user_msg},        # one API call per (call, speech-bucket)
    ],
    temperature=0.2,        # randomness: low -> near-deterministic; 0.0 would be fully greedy
    top_p=0.9,              # nucleus sampling: smallest token set whose probabilities sum to 0.9
    seed=42,                # fixed seed -> repeatable draws where the provider honours it
    max_tokens=400,         # cap the reply; the JSON record is short
    response_format={'type': 'json_object'},            # force strictly parseable JSON, no prose
)

# ### Other providers use the identical OpenAI-style call; swap only the client and model string:
#   OpenAI    : from openai import OpenAI;  client = OpenAI();  model = 'gpt-5.6'
#   DeepSeek  : client = OpenAI(base_url='https://api.deepseek.com');  model = 'deepseek-chat'
#   Groq 8B   : model = 'llama-3.1-8b-instant'   # cheaper; used for the full case-study run
#   Anthropic : from anthropic import Anthropic; Anthropic().messages.create(...) also exposes top_k

import json
scores = json.loads(resp.choices[0].message.content)   # the model's raw JSON answer
print({k: scores[k] for k in ['managerial_confidence', 'specificity',
                              'surprise_negative', 'macro_vs_firm_risk_framing', 'regime_tag']})

{'managerial_confidence': 0.7, 'specificity': 0.8, 'surprise_negative': -0.2,
 'macro_vs_firm_risk_framing': 0.4, 'regime_tag': 'in_line'}


The model reads Coca-Cola's prepared remarks as confident (0.7) and specific (0.8), with a mildly positive surprise (a `surprise_negative` of -0.2, i.e. marginally more upbeat than the prior quarter), risk framed as firm-specific rather than macro (+0.4), and tags the quarter `in_line`. Alongside the four scores the model returns a one-sentence rationale ("management is confident in their ability to deliver solid results ... however, they also acknowledge challenges in North America and Japan and express caution on currency"), which is worth persisting: it makes every score auditable after the fact and surfaces exactly the hedged, forward-looking language a dictionary cannot see.

The decoding parameters are what turn this call into a *reproducible* factor rather than a one-off demonstration. `temperature` (0.2) controls randomness: a low value makes the model near-deterministic, so re-scoring the same transcript does not perturb the cross-section, and 0.0 would make it fully greedy. `top_p` (0.9) is *nucleus sampling*, restricting each token draw to the smallest set of candidates whose probabilities sum to 0.9; with a low temperature it has little further effect, but it is the standard second lever when a provider does not expose `top_k`, the older alternative that instead caps the candidate set at a fixed count (Anthropic's native API exposes `top_k`, while the OpenAI-compatible chat endpoint used here does not). A fixed `seed` makes the pseudo-random draws repeatable on providers that honour it. `response_format={'type': 'json_object'}` forces the model to emit strictly parseable JSON instead of a chatty paragraph, which is what lets a run over tens of thousands of calls proceed without a fragile text parser, and a small `max_tokens` cap is safe because the answer is a short record, not an essay. Because Groq's endpoint is OpenAI-compatible, the identical request runs against OpenAI, DeepSeek, or another compatible provider by swapping only the client constructor and model string, as the commented alternatives show. The full case-study composite below is produced by exactly this prompt, run at scale with the cheaper Llama-3.1-8B variant across the whole NLP US universe.

To demonstrate how LLM-derived sentiment integrates into a factor model, we show the workflow for constructing a sentiment feature on the real case-study output. The construction reuses the case-study panel of Chapter 24 (`mlfi_us_data` merged with the earnings-call sentiment scores): real Llama-3.1-8B four-axis scores already produced on the NLP US universe across 83 quarters of earnings calls, aggregated to a single composite `s_llm_4d` per stock-quarter, broadcast to monthly with a three-month forward-fill, and uniformized cross-sectionally.

In [ ]:
# Workflow: LLM sentiment -> factor signal -> merge with mlfi_us_data
import pandas as pd
panel = pd.read_parquet('data/case_study_panel.parquet')  # the Chapter 24 case-study panel
panel['llm_sentiment'] = panel.groupby('date')['s_llm_4d'].transform(
    lambda x: x.rank(pct=True))                      # uniformize per month
print(f"LLM sentiment feature added: "
      f"n={panel['llm_sentiment'].notna().sum():,}, "
      f"mean={panel['llm_sentiment'].mean():.3f}, "
      f"std={panel['llm_sentiment'].std():.3f}")

LLM sentiment feature added: n=83,078, mean=0.501, std=0.289


The uniformized LLM sentiment feature has the same distributional properties as the other features in the book's dataset (mean exactly 0.5, distributed approximately uniformly on $[0, 1]$), making it directly compatible with the cross-sectional models of Chapters 5 through 9. The 83,078-row coverage is the joint constraint of (a) the NLP US universe × 83-quarter scoring panel of the case study, (b) the data_ml universe match, and (c) the requirement that R1M_Usd is observed. The 421 tickers covered are the large-cap subset for which LLM analyst coverage is dense enough to produce a usable monthly signal; outside this subset the chapter's standard imputation conventions (median-by-date) would have to be invoked.

## Summarization and information extraction

Earnings call transcripts are typically 5,000 to 10,000 words long, containing a prepared management presentation followed by an analyst Q&A session. LLMs can condense these into structured summaries highlighting:

- Key financial metrics and guidance changes.
- Management's assessment of current conditions and outlook.
- Risks and uncertainties mentioned.
- Analyst concerns (revealed through the questions asked).
- Tone shifts between the prepared remarks and the Q&A.

This structured extraction transforms unstructured text into a format suitable for quantitative analysis. For example, one might construct features such as "guidance revision direction" (up, down, or unchanged), "number of risk warnings," or "management confidence score" and include them in a factor model.

Two empirical studies bracket the shift from dictionary-and-ML pipelines to LLM-based extraction and together tell a coherent story about earnings-call predictability. @zhao2018alpha ran a pre-LLM analysis of Russell 3000 earnings calls between May 2010 and December 2017 and backtested three sentiment specifications (level, year-on-year change, and trend) that generated long-short annualised returns of roughly 4.14%, 3.07%, and 3.96%, respectively, all significant at the 1% level. Behavioural features extracted from the same calls (linguistic simplicity and numerical content) generated additional spreads of 2.11% and 4.43% per year. Looking inside the call, the author documents a sharp asymmetry: within management speakers, CEO sentiment alone delivered 3.63% per year and effectively subsumed the contribution of the CFO, while analyst sentiment on its own produced 4.24%. The single most consequential finding is that the Q&A section drives almost all of the predictive power, while the prepared remarks add very little.

@chiang2025boardroom revisit the same question on an extended universe (Russell 3000 from January 2008 through September 2024, roughly 192,000 calls) with LLM-era tooling. They use snowflake-arctic-embed-m to encode each sentence and Llama 3.1 8B (temperature 0) to score every Q&A exchange on two LLM-defined axes: *on-topic alignment*, which measures whether management's answer addresses the question actually asked, and *proactiveness*, which captures whether management volunteers forward-looking context beyond the narrow question. Sector-neutral quintile sorts deliver 390 bps per year on the on-topic signal alone, 170 bps on proactiveness alone, and 506 bps per year on their equal-weighted combination, with the authors attributing the alpha to realised gross-profit-growth differentials (31% for the top quintile against 12% for the bottom quintile) rather than to multiple expansion. Read in sequence, the two papers document that the *signal* (linguistic content of the Q&A) was already there in 2018, and that LLMs allow researchers to extract finer dimensions of it than term-frequency or sentence-level classifiers could.

Neither the Zhao nor the Chiang decomposition can be reproduced exactly here, since both rest on proprietary transcript panels, but the claim they share, that the analyst-Q&A segment carries at least as much signal as the scripted remarks, is exactly the prepared-versus-Q&A comparison that TABLE 21.1 of Chapter 21 reports on the book's universe. There, at the five-day horizon, the analyst-Q&A bucket scores higher than the prepared-remarks bucket for the contextual and general-purpose methods (FinBERT and VADER) and is close behind for the tone dictionaries, so the ordering the two studies document survives on our data. The practical message is the one both papers make: the Q&A section of a conference call is not a residual or a disclosure ritual, it is a primary carrier of the forward-looking information that equity-factor researchers are trying to capture.

## Thematic classification and factor discovery

LLMs can classify news articles and filings by investment theme with minimal supervision. A simple prompt such as "Classify the following news article into one of these investment themes: AI/Technology, Clean Energy, Geopolitical Risk, Consumer Spending, Interest Rates, Healthcare Innovation" produces useful thematic tags without the need for labeled training data.

This capability enables the construction of *thematic baskets*: portfolios of stocks linked to a specific investment theme as identified by text analysis. Traditional approaches require manual curation of company lists (e.g., which firms are "AI-exposed"), while LLM-based approaches can dynamically update theme assignments as the textual evidence changes.

More ambitiously, LLMs can be used for unsupervised factor discovery. By embedding a large corpus of financial texts using the model's internal representations and clustering the resulting vectors, researchers can discover latent factors that emerge from the data rather than being imposed by the researcher. This approach echoes the philosophy of @gu2018empirical, who advocated for data-driven factor discovery using machine learning on numerical characteristics. The extension to textual data is not merely hypothetical: @ke2019predicting build precisely such a model, a supervised text-mining framework (SESTM) that isolates return-predictive terms by predictive screening, weights them through topic modelling, and aggregates them into an article-level score by penalized likelihood. On the Dow Jones Newswires they show that the resulting signal is assimilated into prices with a limits-to-arbitrage delay, more severe for smaller and more volatile firms, that a real-time strategy can exploit net of transaction costs. SESTM is supervised rather than unsupervised, but it is the rigorous precedent for treating text as the raw material of data-driven return prediction, of which the embedding-and-clustering route above is the unsupervised counterpart.

A more direct and rapidly maturing line of work uses the LLM not to embed text but to *write the factors themselves*. The template is human-AI interactive alpha mining, introduced by @wang2023alphagpt in Alpha-GPT: the researcher expresses a trading idea in natural language, the LLM translates it into a formulaic factor expression over price and fundamental data, the factor is backtested, and the numerical feedback is fed back to the model to sharpen the next proposal. This closes a loop that the genetic-programming alpha search of Chapter 19 leaves open, replacing blind mutation of factor expressions with a proposer that carries economic priors about what a sensible factor looks like. Subsequent systems automate the loop end to end. @tang2025alphaagent build AlphaAgent, an autonomous alpha-mining agent whose central concern is *alpha decay*: because an unconstrained model tends to rediscover crowded, overfit signals, they add originality and complexity regularizers that push the search toward factors that are both novel and parsimonious, and report backtests in which the mined factors retain predictive power longer than genetic-programming baselines. @shi2026alphajungle recast the same formulaic-alpha search as a Monte Carlo tree search guided by an LLM, so the model both proposes candidate expressions and refines them through reflective feedback, a natural marriage of the deliberative tree search of Section 23.11 with the factor-expression space of Chapter 19. @kou2024automate embed factor generation inside a risk-aware multi-agent pipeline, in which prompt-engineered models generate executable factor candidates across heterogeneous data sources, a filtering agent screens them on predictive quality and category balance, and a dynamic optimizer reweights the survivors by market regime. Read together, this literature is the LLM-era successor to the data-driven factor construction of @gu2018empirical and the evolutionary alpha search of Chapter 19: the model supplies the economic priors and the code-generation ability that a blind search lacks, but the overfitting and multiple-testing hazards catalogued in Section 23.13 apply with full force, and none of these systems should be trusted without the out-of-sample discipline that the rest of this book insists on.

## LLM agents for financial research

The applications surveyed in Sections 23.8 through 23.10 all rely on a *single-prompt* paradigm: a chunk of text in, a labelled or summarised answer out. The most recent direction in the literature dissolves this constraint by treating the LLM as the reasoning core of a larger *agent* that can plan, call tools, write code, retrieve from external memory, and iterate. For factor investing the appeal is the natural fit between a long-running research workflow (ingest a filing, extract the relevant numbers, compare to peers, score against a factor library, decide on a position) and an agent's stepwise reasoning loop.

![FIGURE 23.2: An LLM-agent architecture for financial research](images/figure_23_2_llm_agent_architecture.png)

FIGURE 23.2 sketches the canonical agent template. Four ingredients distinguish it from a plain LLM call: a *planner* that decomposes the task and decides what to do next, a *reasoning core* (often a reasoning-tuned LLM running chain-of-thought as discussed in Section 23.5), a *tool layer* through which the agent invokes external capabilities (data fetchers, quant libraries, code interpreters, RAG retrieval, other agents), and a *memory layer* with short-term context, mid-term episodic recall, and long-term semantic storage.

**Foundations.** The agent paradigm rests on three foundational papers. @yao2023react introduced ReAct, a prompting pattern that interleaves chain-of-thought reasoning steps with explicit action calls (search, calculate, lookup), producing trajectories of the form

$$
\tau = (s_0,\, \underbrace{t_0, a_0, o_1}_{\text{thought-act-observe}},\, t_1, a_1, o_2,\, \ldots,\, t_{T-1}, a_{T-1}, o_T,\, y)
\tag{23.5}
$$

where $s_0$ is the task prompt, $t_k$ is a free-form reasoning step ("thought"), $a_k \in \mathcal{A}$ is a discrete action (a tool call, with $\mathcal{A}$ a finite vocabulary of tool names), $o_{k+1}$ is the environment's observation returned by that tool, and $y$ is the final answer. The trajectory is auditable end-to-end (each $t_k$ and $a_k$ is a token sequence emitted by the LLM, each $o_{k+1}$ is a tool return value), which is precisely what makes ReAct attractive for regulated financial workflows: every step of the agent's reasoning is logged and replayable. @schick2023toolformer showed in Toolformer that LLMs can themselves learn when and how to call tools by self-supervised insertion of API calls into pretraining text, eliminating the need for hand-crafted prompts. @du2023multiagent demonstrated that having multiple LLM instances debate and critique each other improves factual accuracy on reasoning benchmarks compared to a single instance, motivating the multi-agent system architectures (MAS) that now dominate agentic-research products.

A further refinement bridges single-trajectory ReAct and multi-agent debate by treating the agent's deliberation as a search problem in its own right. @zhou2024lats introduced Language Agent Tree Search (LATS), which combines Monte Carlo tree search with an LLM-based planner: the agent simulates and scores multiple action sequences, backpropagating reward estimates to refine its choice of next step. In a financial-research setting the appeal is that LATS naturally accommodates the explore-versus-exploit tradeoff that arises whenever an analyst must choose between probing one filing in depth or scanning a wider cross-section, and the framework is among the methods surveyed in the practitioner guide *AI Agents: The Definitive Guide* [@koenigstein2025agents].

**Finance-specific agent frameworks.** A small but rapidly growing literature instantiates this template for trading and analyst workflows. @yu2024finmem introduced FinMem, an LLM trading agent that organises its memory into the short-term, mid-term, and long-term layers shown in FIGURE 23.2, and that uses a configurable *character* prior (risk-averse, momentum-following, value-oriented) to bias the policy; on a multi-asset backtest FinMem improves on plain GPT-4 prompting both in cumulative return and in the consistency of the realised strategy. @li2023tradinggpt proposed TradingGPT, a multi-agent system in which several LLM personae (a fundamentals analyst, a technical analyst, a risk officer) debate in the style of @du2023multiagent before a head agent commits to a trade. @yang2024finrobot released FinRobot as an open-source agent platform that wires general-purpose LLMs into the data-fetching, reporting, and recommendation tasks that constitute the practitioner's analyst pipeline, providing a reproducible reference implementation that academic researchers can extend. More recent systems push the multi-agent idea further. @yu2024fincon introduce FinCon, a manager-analyst hierarchy in which a conceptual verbal-reinforcement channel propagates risk-control lessons between trading episodes, improving both single-stock trading and portfolio allocation over a flat prompting baseline. @xiao2024tradingagents organise TradingAgents explicitly around a trading firm's org chart, with fundamental, sentiment, and technical analysts, a trader, and a risk manager engaging in the structured debate of @du2023multiagent before a decision is committed. @zhang2024finagent extend the paradigm to multimodality in FinAgent, a tool-augmented trading agent that ingests news, prices, and candlestick-chart images together, a reminder that the agent's tool layer can reach beyond text to the visual data that human analysts also consult.

**Practical guidance and caveats.** Agents are powerful and, for factor research, also expensive. Each LLM call inside the agent is metered, and a single multi-step workflow can rack up tens of thousands of tokens; we discuss the economics of this in Chapter 24. Multi-agent systems compound the cost further, with linear scaling in the number of personae for any non-trivial debate length. Three caveats temper the enthusiasm. First, much of the published agent literature reports gains on benchmarks that do not match factor-research workflows, so the headline numbers in FinMem and TradingGPT papers should be read as upper bounds rather than realistic expectations. Second, agents fail in characteristic ways: they over-execute (call too many tools), under-explore (commit to a path early), or hallucinate inputs to tool calls; building a robust agent in practice means building extensive logging and unit-tests around its tool surface. Third, the alignment problem (Section 23.5) compounds in agent settings, because errors propagate through tool calls rather than being contained in a single response. Practitioners deploying agentic systems in real trading workflows therefore typically restrict the agent's tool surface (read-only data access, no order-routing) and place a human-in-the-loop checkpoint before any externally observable action. None of this should discourage experimentation; agents are the most natural fit for the multi-step structure of factor research that we have yet seen, and the pace of progress in 2024 to 2026 has been rapid. The fair summary is that LLM agents are now ready for the *research-and-prototyping* part of the pipeline, and approaching readiness for the *production* part.

## Evaluating LLM analysts: benchmarks and reality checks

The applications surveyed in Sections 23.8 through 23.11 raise a natural question: how good are these systems, really, at the financial tasks we want them to perform? A young but increasingly serious literature on *finance-specific evaluation benchmarks* allows practitioners to calibrate expectations against published evidence rather than vendor demos. We highlight three benchmark families that have become standard reference points.

**PIXIU and FLARE.** @xie2023pixiu released PIXIU, a finance-instruction dataset of 136,000 supervised examples spanning five tasks (sentiment classification, headline classification, named entity recognition, question answering, and stock-movement prediction) together with FLARE, the matched evaluation suite. They use PIXIU to fine-tune FinMA, an open-source 7B-parameter LLaMA derivative, and benchmark it head-to-head against BloombergGPT, FinBERT, and ChatGPT. The headline finding is that domain-specific instruction tuning materially closes the gap between a small open model and a much larger closed one on most NLP tasks, but the gap reopens sharply on prediction tasks where multi-step reasoning is required. PIXIU was the first widely-adopted public finance-LLM benchmark and remains the most-cited reference in the space.

**FinBen.** @xie2024finben, in a NeurIPS 2024 follow-up, generalise PIXIU to FinBen, a holistic benchmark spanning 36 datasets across 24 tasks: information extraction, textual analysis, question answering, text generation, risk management, forecasting, and decision-making. Their evaluation of GPT-4, ChatGPT, FinMA, and a panel of open-source LLMs documents a clean stratification by task class: information extraction is largely solved, sentiment-style classification is approaching ceiling, question answering depends heavily on retrieval quality, and *forecasting and decision-making remain far from solved* with the strongest model (GPT-4) achieving only modest accuracy gains over chance on multi-step financial reasoning tasks. The result anchors the cautious tone we have used throughout this chapter when discussing LLM-derived alphas.

**FinanceBench.** @islam2023financebench take a different angle: rather than benchmarking representative tasks at scale, they construct a carefully curated open-source evaluation subset of 150 expert-annotated financial questions over public filings (drawn from a full corpus of 10,231 questions) and measure how well retrieval-augmented LLMs do on them. The headline number is sobering and configuration-dependent. With a shared vector store spanning all filings (the weakest retrieval configuration), GPT-4-Turbo refuses to answer or answers incorrectly on **81%** of FinanceBench questions; with a per-document vector store the failure rate falls to roughly **50%**, and with the full document in the context window it falls further to about **21%**. For less capable open-source models the aggregate failure rate exceeds 90%. The paper's contribution is to demonstrate, on a tightly-defined task, that retrieval quality dominates the apparent LLM-capability gap and that the gap between LLM marketing claims and reliable financial QA performance remains very large under realistic retrieval constraints.

Taken together, the PIXIU, FinBen, and FinanceBench results tell a coherent story: LLMs are now reliably useful for extraction, classification, and summarisation over financial text, but the harder tasks (multi-step reasoning over filings, forecasting, decision-making, and any task requiring numerical accuracy under retrieval failure) remain unsolved. For factor investors, this means LLM-derived features are best deployed as *signal complements* to traditional factors (Section 21.7) rather than as standalone prediction engines, and any production deployment should include a regression test against the relevant subset of these benchmarks to monitor for capability regressions and data leakage.

## Current limitations

Several challenges limit the current usefulness of NLP in factor investing.

**Data snooping and the factor zoo.** @harvey2016and warned that the proliferation of proposed factors (over 400 at the time of their study) makes it likely that many are false discoveries. NLP-derived factors are particularly susceptible to this concern because text data offers enormous flexibility in signal construction: the researcher can choose the text source, preprocessing pipeline, sentiment model, aggregation method, and rebalancing frequency. Without rigorous multiple-testing corrections, such as the Deflated Sharpe Ratio proposed by @bailey2014deflated, it is easy to data-mine spurious NLP factors.

**Language drift.** NLP models trained on pre-2020 text may not accurately process post-pandemic language, which introduced terms like "social distancing," "supply chain bottleneck," and "meme stock" into financial discourse. Sentiment dictionaries are particularly vulnerable to drift because they are static; transformer models are more robust but still degrade as language evolves. Periodic retraining or re-calibration is necessary.

**Coverage bias.** Large-cap stocks receive far more news coverage, analyst reports, and social media attention than small-cap stocks. NLP-derived factors therefore have higher signal quality for large caps and may be unreliable for the long tail of the cross-section. This creates a selection bias that must be addressed in portfolio construction (e.g., by restricting the investment universe to stocks with adequate text coverage).

**Reproducibility.** LLM outputs are stochastic (due to temperature sampling) and can change between model versions. A sentiment score generated by GPT-4 may differ from one generated by GPT-4-turbo or GPT-5. This complicates backtesting and replication, fundamental requirements of rigorous quantitative research.

## The road ahead

Several emerging trends are likely to shape the future of NLP in factor investing. The list below groups them into three families: *modality and language expansion* (multimodal, multilingual, real-time), *human-machine collaboration* (AI copilots), and *signal-persistence considerations* (alpha decay).

**Multimodal models.** Text is only one modality of unstructured data. Earnings calls include audio (voice tone, hesitation, emotion), filings include tables and charts, and corporate communications include images and video. Multimodal models that jointly process text, audio, tabular data, and images are beginning to emerge and may capture richer signals than text alone.

**Multilingual NLP.** Most financial NLP research focuses on English-language text. For global equity investing, the ability to process text in Chinese, Japanese, German, and other languages is essential. Multilingual transformer models (such as mBERT and XLM-R) provide a foundation, but domain-specific adaptation for financial text in non-English languages remains an open area.

**Real-time event detection.** The boundary between factor investing (monthly rebalancing) and high-frequency trading (millisecond execution) is blurring. Real-time NLP pipelines that detect market-moving events (earnings surprises, M&A announcements, geopolitical shocks) within seconds of publication could enable faster factor signal updates.

**AI copilots for fundamental analysts.** Rather than replacing human analysts, LLMs are increasingly being deployed as *copilots* that assist with document summarization, data extraction, and hypothesis generation. This human-AI collaboration model may prove more robust than fully automated approaches, particularly for complex, ambiguous situations that require judgment.

**Will NLP alpha persist?** The question is a special case of the broader crowding debate: as a data class is more widely adopted, the informational advantage it provides should erode, consistent with the efficient market hypothesis. We think the mechanical version of that claim, that any "in vogue" dataset must lose value as more market participants discover it, is too crude. The erosion is conditional on at least two things, each of which is materially less binding for NLP than the simple crowding story assumes. The first is the *obviousness* of the data: how unambiguously can a participant move from the raw input to a signal, and from a signal to an executed trade? With prices and fundamentals the path is short and standardised, which is why crowding pressure on classical factor strategies is real and measurable. With text the path is forked at every step. A practitioner can score a transcript with a plain off-the-shelf LLM call, with a domain-fine-tuned FinBERT, with a custom-trained encoder over a proprietary corpus, or with an LLM-as-agent pipeline of the kind discussed in Section 23.11, and each of these choices generates a different signal with different cross-sectional dispersion, different turnover, and different correlation with classical factors (the LM-versus-FinBERT-versus-LLM gap quantified in TABLE 21.1 of Chapter 21 is precisely an instance of this multiplicity). NLP remains squarely within the family of alternative data sources whose obviousness is markedly lower than that of prices or fundamentals, and the empirical evidence reviewed throughout this chapter shows that the dispersion across implementation choices is large enough to keep meaningful return-predictive content alive even as the *broad* technology is adopted. The second conditional is trading style and execution: signals do not capture returns by themselves, the returns accrue to participants who can act on them at sufficient speed, scale, and at acceptable transaction-cost levels. A daily-rebalanced cross-sectional sentiment factor, a five-day post-call event-window strategy, and a high-frequency tweet-based macro strategy all extract value from "NLP signal" in principle, but their realised alpha depends on whether the implementing desk has the latency, the borrow availability, the venue selection, and the capacity to translate the signal into a trade before the rest of the market does. Both conditionals push the persistence question away from a mechanical "more participants implies less alpha" inevitability and toward a richer empirical question that is closer to the trajectory of any other live research area in factor investing.

## Coding exercises

1. This chapter argues that, whenever an LLM score is destined for a
   *cross-sectional* sort, the most consequential line in the prompt is the
   calibration anchor that fixes what a score of zero means (Section 23.8). Test
   the claim on data you can gather yourself. Using the free SEC EDGAR full-text
   search, download the Risk Factors section (Item 1A) from the most recent 10-K
   of at least ten large-cap firms of your choice, and score each disclosure on a
   single continuous forward-risk-severity axis, calibrated so that a typical
   large-cap filing scores zero, using a free LLM endpoint (for example the Groq
   free tier) or a local open-source model. Rerun the scoring with the
   calibration anchor removed from the prompt, compare the cross-sectional
   standard deviation of the two score sets, and explain which version you would
   feed into a factor sort and why. The exercise needs no proprietary return
   data: it stops at the scoring step, which is where the modelling choices this
   chapter emphasises actually live.

## References

```{bibliography}
```